# TTM Zero-Shot Error Metric Analysis

Computes **MSE, RMSE, MAE, MASE, MAPE, sMAPE** from the saved outputs in `ttm_results_v1/`.

- Predictions and actuals are **inverse-scaled** before computing metrics.
- For each site × pollutant the error is the **mean over the 12 prediction steps**, then averaged across all rolling windows.
- MASE uses a naïve one-step-ahead baseline computed on the context (past) values.
- Final output: one row per *site × pollutant* with all six metrics.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import pickle as pkl
from tqdm import tqdm

RESULTS_DIR = "/home/student/rishi/ttm_results_v1"
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 12

## Helper Functions — Inverse Scaling & Error Metrics

In [2]:
def inverse_scale(arr: np.ndarray, mean: np.ndarray, scale: np.ndarray) -> np.ndarray:
    """Inverse standard-scaling: x_orig = x_scaled * scale + mean.
    
    Args:
        arr: (..., n_channels) scaled values
        mean: (n_channels,) per-channel means
        scale: (n_channels,) per-channel stds
    Returns:
        Inverse-scaled array with same shape as `arr`.
    """
    return arr * scale + mean

# ── Per-window metric helpers (vectorised over the horizon axis) ────
def mse_pw(a, p):
    """(N,) MSE per window, averaged over horizon."""
    return np.mean((a - p) ** 2, axis=1)

def rmse_pw(a, p):
    return np.sqrt(mse_pw(a, p))

def mae_pw(a, p):
    return np.mean(np.abs(a - p), axis=1)

def mape_pw(a, p, eps=1e-8):
    """(N,) MAPE per window. Windows where all |actual| < eps → NaN."""
    mask = np.abs(a) > eps
    # per-element percentage errors (masked entries set to 0)
    pct = np.where(mask, np.abs((a - p) / a), 0.0)
    counts = mask.sum(axis=1)
    with np.errstate(invalid="ignore"):
        out = pct.sum(axis=1) / counts * 100
    out[counts == 0] = np.nan
    return out

def smape_pw(a, p, eps=1e-8):
    denom = np.clip((np.abs(a) + np.abs(p)) / 2.0, eps, None)
    return np.mean(np.abs(a - p) / denom, axis=1) * 100

def mase_pw(a, p, past, eps=1e-12):
    """(N,) MASE per window using naïve 1-step baseline on context."""
    naive_mae = np.mean(np.abs(np.diff(past, axis=1)), axis=1)  # (N,)
    mae_vals = np.mean(np.abs(a - p), axis=1)                   # (N,)
    with np.errstate(invalid="ignore"):
        out = mae_vals / naive_mae
    out[naive_mae < eps] = np.nan
    return out

## Compute Metrics for Every Site

Iterates over all site folders, loads the three artefacts, inverse-scales, and computes metrics per pollutant.

In [3]:
rows = []

site_dirs = sorted(
    d for d in os.listdir(RESULTS_DIR)
    if os.path.isdir(os.path.join(RESULTS_DIR, d))
)


for site_name in tqdm(site_dirs, desc="Computing per-window metrics"):
    site_path = os.path.join(RESULTS_DIR, site_name)

    # ── Load artefacts ──────────────────────────────────────────────
    dataset = torch.load(os.path.join(site_path, "dataset.pt"), weights_only=False)
    preds_tensor = torch.load(os.path.join(site_path, "predictions.pt"), weights_only=False)
    with open(os.path.join(site_path, "scaler_params.pkl"), "rb") as f:
        scaler = pkl.load(f)

    future_vals = dataset["future_values"].numpy()   # (N, 12, 6)
    past_vals   = dataset["past_values"].numpy()      # (N, 512, 6)
    preds_np    = preds_tensor.numpy()                 # (N, 12, 6)

    mean_ = np.array(scaler["mean_"])                  # (6,)
    scale_ = np.array(scaler["scale_"])                # (6,)
    target_columns = scaler["target_columns"]          # list of 6

    # ── Inverse-scale everything ────────────────────────────────────
    future_inv = inverse_scale(future_vals, mean_, scale_)   # (N, 12, 6)
    preds_inv  = inverse_scale(preds_np, mean_, scale_)      # (N, 12, 6)
    past_inv   = inverse_scale(past_vals, mean_, scale_)     # (N, 512, 6)

    N = future_inv.shape[0]

    # Start a dict for this site's rows
    site_data = {
        "site":              [site_name] * N,
        "window_idx":        np.arange(N),
        "context_length":    [CONTEXT_LENGTH] * N,
        "prediction_length": [PREDICTION_LENGTH] * N,
    }

    # ── Metrics per pollutant → separate columns ────────────────────
    for col_idx, col_name in enumerate(target_columns):
        a = future_inv[:, :, col_idx]   # (N, 12)
        p = preds_inv[:, :, col_idx]    # (N, 12)
        c = past_inv[:, :, col_idx]     # (N, 512)

        site_data[f"{col_name}_MSE"]   = mse_pw(a, p)
        site_data[f"{col_name}_RMSE"]  = rmse_pw(a, p)
        site_data[f"{col_name}_MAE"]   = mae_pw(a, p)
        site_data[f"{col_name}_MASE"]  = mase_pw(a, p, c)
        site_data[f"{col_name}_MAPE"]  = mape_pw(a, p)
        site_data[f"{col_name}_sMAPE"] = smape_pw(a, p)

    rows.append(pd.DataFrame(site_data))

metrics_df = pd.concat(rows, ignore_index=True)
print(f"Shape: {metrics_df.shape}")
metrics_df.head(10)

Computing per-window metrics:  83%|████████▎ | 115/138 [02:42<00:32,  1.41s/it]/tmp/ipykernel_1236182/3429071577.py:44: RuntimeWarning: divide by zero encountered in divide
  out = mae_vals / naive_mae
Computing per-window metrics: 100%|██████████| 138/138 [03:14<00:00,  1.41s/it]


Shape: (3557778, 40)


,site,window_idx,context_length,prediction_length,PM2.5 (µg/m³)_MSE,PM2.5 (µg/m³)_RMSE,PM2.5 (µg/m³)_MAE,PM2.5 (µg/m³)_MASE,PM2.5 (µg/m³)_MAPE,PM2.5 (µg/m³)_sMAPE,...,CO (mg/m³)_MAE,CO (mg/m³)_MASE,CO (mg/m³)_MAPE,CO (mg/m³)_sMAPE,Ozone (µg/m³)_MSE,Ozone (µg/m³)_RMSE,Ozone (µg/m³)_MAE,Ozone (µg/m³)_MASE,Ozone (µg/m³)_MAPE,Ozone (µg/m³)_sMAPE
0,site_113_Shadipur_Delhi_CPCB_15Min,0,512,12,1197.838554,34.609804,29.963838,0.715483,14.220257,13.910127,...,0.160951,1.039576,33.914773,23.619950,142.018887,11.917168,10.678458,1.418005,30.356178,36.567469
1,site_113_Shadipur_Delhi_CPCB_15Min,1,512,12,1172.761373,34.245604,29.983297,0.716108,14.045259,13.608555,...,0.162382,1.047414,34.757682,23.889076,135.308215,11.632206,10.841964,1.439334,32.009980,38.665168
2,site_113_Shadipur_Delhi_CPCB_15Min,2,512,12,834.979317,28.896009,23.457284,0.559919,11.433359,10.622976,...,0.172004,1.113794,37.644089,24.944399,56.476476,7.515083,6.672236,0.881475,22.637778,26.798744
3,site_113_Shadipur_Delhi_CPCB_15Min,3,512,12,1865.668691,43.193387,31.702145,0.756651,15.142003,13.421901,...,0.166130,1.075484,36.325602,24.294197,58.986633,7.680276,7.002373,0.923929,24.833006,29.486605
4,site_113_Shadipur_Delhi_CPCB_15Min,4,512,12,2257.635274,47.514580,36.789875,0.879426,17.077992,15.097932,...,0.155571,1.014713,33.762722,22.972267,56.097982,7.489859,7.079425,0.933526,28.832111,31.597443
5,site_113_Shadipur_Delhi_CPCB_15Min,5,512,12,3283.810312,57.304540,46.846178,1.119854,22.297743,19.127316,...,0.149385,1.005027,32.129576,21.886926,48.910373,6.993595,6.542553,0.862359,30.403776,31.671522
6,site_113_Shadipur_Delhi_CPCB_15Min,6,512,12,2738.915025,52.334645,43.042613,1.027675,20.442168,17.940511,...,0.153128,1.043746,32.294952,22.239791,47.220596,6.871724,6.409317,0.845328,34.013072,33.949209
7,site_113_Shadipur_Delhi_CPCB_15Min,7,512,12,2662.009579,51.594666,46.002961,1.097412,21.429504,19.517492,...,0.162106,1.101836,36.149449,23.179088,37.449117,6.119568,5.631693,0.742760,36.002827,33.909832
8,site_113_Shadipur_Delhi_CPCB_15Min,8,512,12,2782.299589,52.747508,47.927331,1.143826,22.656810,20.513362,...,0.165801,1.201101,35.896656,23.753158,35.586342,5.965429,5.480796,0.723251,35.875848,33.535304
9,site_113_Shadipur_Delhi_CPCB_15Min,9,512,12,2647.636374,51.455188,48.142157,1.147276,22.054013,21.196989,...,0.162753,1.255691,30.685572,23.009107,30.231155,5.498287,5.017565,0.664338,36.594455,32.356908


## Summary Statistics by Pollutant

Average metrics across all sites for each pollutant.

In [4]:
# Build a long-form view for easy grouping by pollutant
metric_names = ["MSE", "RMSE", "MAE", "MASE", "MAPE", "sMAPE"]
pollutant_cols = [c for c in metrics_df.columns if any(c.endswith(f"_{m}") for m in metric_names)]
pollutants = sorted({c.rsplit("_", 1)[0] for c in pollutant_cols})

summary_rows = []
for poll in pollutants:
    row = {"pollutant": poll}
    for m in metric_names:
        col = f"{poll}_{m}"
        if col in metrics_df.columns:
            row[f"{m}_mean"]   = metrics_df[col].mean()
            row[f"{m}_median"] = metrics_df[col].median()
            row[f"{m}_std"]    = metrics_df[col].std()
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index("pollutant").round(4)
summary

,MSE_mean,MSE_median,MSE_std,RMSE_mean,RMSE_median,RMSE_std,MAE_mean,MAE_median,MAE_std,MASE_mean,MASE_median,MASE_std,MAPE_mean,MAPE_median,MAPE_std,sMAPE_mean,sMAPE_median,sMAPE_std
pollutant,,,,,,,,,,,,,,,,,,
CO (mg/m³),0.2281,0.0377,1.1192,0.3050,0.1941,0.3675,0.2469,0.1586,0.2956,1.7430,1.2714,2.3616,2.464586e+07,23.2209,2.143535e+08,29.8871,22.6166,26.2960
NO2 (µg/m³),140.4552,15.1865,865.8816,6.8721,3.8970,9.6555,5.5937,3.1833,7.7717,1.6918,1.1146,4.7142,3.574610e+01,16.9315,2.676946e+02,21.0917,16.6310,19.0204
Ozone (µg/m³),200.6615,43.2667,754.0724,9.6353,6.5777,10.3837,7.8750,5.3614,8.4111,1.5782,1.1567,2.1222,8.851190e+01,26.6692,5.380755e+02,32.4813,25.5076,26.1180
PM10 (µg/m³),3329.1722,747.3005,9472.1578,40.1569,27.3368,41.4318,32.7454,22.6058,33.0305,1.6577,1.3107,1.4189,3.994870e+01,22.9778,2.710548e+02,26.9536,22.4899,18.0621
PM2.5 (µg/m³),1094.7914,156.5620,6384.4107,20.0399,12.5125,26.3286,16.4348,10.3583,20.8211,1.6795,1.3114,1.5915,5.121610e+01,25.8348,3.756614e+02,29.7763,25.0277,19.4308
SO2 (µg/m³),46.9449,4.3168,258.0960,3.8131,2.0777,5.6925,3.0358,1.7120,4.3023,1.7920,1.2545,2.8995,6.734230e+01,20.2547,4.075131e+02,25.8010,19.6982,22.3748


## Export

In [5]:
out_path = "/home/student/rishi/ttm_zeroshot_metrics_w_baseline.csv"
metrics_df.to_csv(out_path, index=False)
print(f"Saved {len(metrics_df)} rows → {out_path}")

Saved 3557778 rows → /home/student/rishi/ttm_zeroshot_metrics_w_baseline.csv


## Hourly Mean & Median Baselines

For each prediction step, the **mean baseline** predicts the mean of all context values at the same hour-of-day, and the **median baseline** uses the median.

Since the data is hourly and `CONTEXT_LENGTH = 512` ($512 \bmod 24 = 8$), prediction step $j$ aligns with context indices where $k \bmod 24 = (8 + j) \bmod 24$ — consistent across all windows.

In [6]:
mean_rows = []
median_rows = []

# Precompute matching context indices for each of the 12 prediction steps
# Since 512 % 24 = 8, prediction step j maps to context hour (8+j) % 24
ctx_idx_per_step = [np.arange((8 + j) % 24, CONTEXT_LENGTH, 24) for j in range(PREDICTION_LENGTH)]

for site_name in tqdm(site_dirs, desc="Computing hourly baselines"):
    site_path = os.path.join(RESULTS_DIR, site_name)

    # ── Load artefacts ──────────────────────────────────────────────
    dataset = torch.load(os.path.join(site_path, "dataset.pt"), weights_only=False)
    with open(os.path.join(site_path, "scaler_params.pkl"), "rb") as f:
        scaler = pkl.load(f)

    future_vals = dataset["future_values"].numpy()   # (N, 12, C)
    past_vals   = dataset["past_values"].numpy()      # (N, 512, C)

    mean_  = np.array(scaler["mean_"])
    scale_ = np.array(scaler["scale_"])
    target_columns = scaler["target_columns"]

    # ── Inverse-scale ────────────────────────────────────────────────
    future_inv = inverse_scale(future_vals, mean_, scale_)
    past_inv   = inverse_scale(past_vals, mean_, scale_)

    N = future_inv.shape[0]

    mean_site = {
        "site": [site_name] * N, "window_idx": np.arange(N),
        "context_length": [CONTEXT_LENGTH] * N, "prediction_length": [PREDICTION_LENGTH] * N,
    }
    median_site = {
        "site": [site_name] * N, "window_idx": np.arange(N),
        "context_length": [CONTEXT_LENGTH] * N, "prediction_length": [PREDICTION_LENGTH] * N,
    }

    for col_idx, col_name in enumerate(target_columns):
        a = future_inv[:, :, col_idx]   # (N, 12) — actual
        c = past_inv[:, :, col_idx]     # (N, 512) — context

        # Build hourly baselines: (N, 12)
        mean_bl   = np.column_stack([np.mean(c[:, idx], axis=1)   for idx in ctx_idx_per_step])
        median_bl = np.column_stack([np.median(c[:, idx], axis=1) for idx in ctx_idx_per_step])

        # ── Mean-baseline metrics ────────────────────────────────────
        mean_site[f"{col_name}_MSE"]   = mse_pw(a, mean_bl)
        mean_site[f"{col_name}_RMSE"]  = rmse_pw(a, mean_bl)
        mean_site[f"{col_name}_MAE"]   = mae_pw(a, mean_bl)
        mean_site[f"{col_name}_MASE"]  = mase_pw(a, mean_bl, c)
        mean_site[f"{col_name}_MAPE"]  = mape_pw(a, mean_bl)
        mean_site[f"{col_name}_sMAPE"] = smape_pw(a, mean_bl)

        # ── Median-baseline metrics ──────────────────────────────────
        median_site[f"{col_name}_MSE"]   = mse_pw(a, median_bl)
        median_site[f"{col_name}_RMSE"]  = rmse_pw(a, median_bl)
        median_site[f"{col_name}_MAE"]   = mae_pw(a, median_bl)
        median_site[f"{col_name}_MASE"]  = mase_pw(a, median_bl, c)
        median_site[f"{col_name}_MAPE"]  = mape_pw(a, median_bl)
        median_site[f"{col_name}_sMAPE"] = smape_pw(a, median_bl)

    mean_rows.append(pd.DataFrame(mean_site))
    median_rows.append(pd.DataFrame(median_site))

mean_baseline_df = pd.concat(mean_rows, ignore_index=True)
median_baseline_df = pd.concat(median_rows, ignore_index=True)
print(f"Mean baseline shape:   {mean_baseline_df.shape}")
print(f"Median baseline shape: {median_baseline_df.shape}")
mean_baseline_df.head(5)

Computing hourly baselines:  83%|████████▎ | 115/138 [07:08<01:24,  3.69s/it]/tmp/ipykernel_1236182/3429071577.py:44: RuntimeWarning: divide by zero encountered in divide
  out = mae_vals / naive_mae
Computing hourly baselines: 100%|██████████| 138/138 [08:34<00:00,  3.73s/it]


Mean baseline shape:   (3557778, 40)
Median baseline shape: (3557778, 40)


,site,window_idx,context_length,prediction_length,PM2.5 (µg/m³)_MSE,PM2.5 (µg/m³)_RMSE,PM2.5 (µg/m³)_MAE,PM2.5 (µg/m³)_MASE,PM2.5 (µg/m³)_MAPE,PM2.5 (µg/m³)_sMAPE,...,CO (mg/m³)_MAE,CO (mg/m³)_MASE,CO (mg/m³)_MAPE,CO (mg/m³)_sMAPE,Ozone (µg/m³)_MSE,Ozone (µg/m³)_RMSE,Ozone (µg/m³)_MAE,Ozone (µg/m³)_MASE,Ozone (µg/m³)_MAPE,Ozone (µg/m³)_sMAPE
0,site_113_Shadipur_Delhi_CPCB_15Min,0,512,12,3079.142202,55.490019,48.438612,1.156628,21.058680,23.579958,...,0.165484,1.068854,31.124848,23.812751,171.702061,13.103513,11.584571,1.538329,33.052826,40.776531
1,site_113_Shadipur_Delhi_CPCB_15Min,1,512,12,2976.883993,54.560828,47.721945,1.139770,20.100307,22.291544,...,0.146119,0.942513,28.793359,21.783661,174.098896,13.194654,11.723262,1.556331,33.956076,42.165760
2,site_113_Shadipur_Delhi_CPCB_15Min,2,512,12,2184.347323,46.737002,40.431826,0.965097,17.122072,18.520372,...,0.148909,0.964244,29.201806,22.216991,117.010105,10.817121,10.233480,1.351954,32.294012,39.111296
3,site_113_Shadipur_Delhi_CPCB_15Min,3,512,12,2197.867246,46.881417,40.532976,0.967421,17.250830,17.981190,...,0.153988,0.996882,29.936622,22.904558,107.917426,10.388331,9.747865,1.286183,32.486337,39.387365
4,site_113_Shadipur_Delhi_CPCB_15Min,4,512,12,2000.634004,44.728447,39.018294,0.932694,16.617597,16.736885,...,0.156667,1.021861,30.285909,23.262478,97.566663,9.877584,9.162091,1.208156,34.809095,39.869743


## Export Baseline Metrics

In [7]:
mean_out   = "/home/student/rishi/mean_baseline_metrics.csv"
median_out = "/home/student/rishi/median_baseline_metrics.csv"

mean_baseline_df.to_csv(mean_out, index=False)
median_baseline_df.to_csv(median_out, index=False)

print(f"Saved {len(mean_baseline_df)} rows → {mean_out}")
print(f"Saved {len(median_baseline_df)} rows → {median_out}")

Saved 3557778 rows → /home/student/rishi/mean_baseline_metrics.csv
Saved 3557778 rows → /home/student/rishi/median_baseline_metrics.csv
